# **Chapter 8 : Gradient Descent**

## **Idea behind Gradient Descent**
### **Optimisation**
We want best solution to our problems . For instance:
- Minimising the error of the model.
- Maximising the likelihood of the data.
Both are optimisation probelms. 

#### **Example:** 
We can try to find a vector that minimises the sum of squares function.

In [1]:
def sum_of_squares(v):
    return sum(v_i **2 for v_i in v)

print(sum_of_squares([1, 2, 3]))

14


### **Gradient Descent**
The gradient is a vector of partial derivatives. It points in the direction where a function increases most rapidly. <br>
For a function of one variable, the derivative tells us the direction of increase or decrease. For a function of several variables, the gradient gives that information across all input dimensions.<br>

To minimise a function, we move in the opposite direction of the gradient.<br>
Basic algorithm says: <br>
 Choose a starting point $\rightarrow$  Calculate the gradient $\rightarrow$ Take small steps opposite the gradient $\rightarrow$ Repeat until the changes become suffieciently small.


### **Estimating the Gradient**
To estimate derivatives numerically, we use a difference quotient.<br>
For a function $f$, the derivative at $x$ can be approximated by
$$
f'(x) \approx  \frac{f(x+h) - f(x)}{h}
$$
Here, $h$ is a small number.

In [5]:
def difference_quotient(f,x,h):
    return (f(x+h) - f(x)) /h

def square(x):
    return x*x
print(difference_quotient(square, 3, 0.0001)) #f'(x)= 2x = 6 and code gives 6.0001.. which is same

6.000100000012054


### **Estimating the Gradient Descent for multiple variables**
#### **1.Partial Difference Quotient**
For a function of multiple variables, we calculate how the function changes when we change only one component of the input vector.



In [2]:
def partial_difference_quotient(f, v, i, h):
    w = [ v_j + (h if j==i else 0)
                 for j, v_j in enumerate(v)
                 ]
    return (f(w) - f(v)) /h

In [3]:
# f(x,y) = x**2 +y**2 and v = [3, 4]
def f(v):
    x, y = v
    return x**2 + y**2

print(partial_difference_quotient(f, [3, 4], 0, 0.00001))
print(partial_difference_quotient(f, [3, 4], 1, 0.00001))

6.000009999951316
8.00000999952033


In [7]:
# estimate the complete gradient
def estimate_gradient(f, v, h=0.00001):
    return [
        partial_difference_quotient(f, v, i, h)
        for i, _ in enumerate(v)
    ]

print(estimate_gradient(f, [3,4]))  #del f(x,y) = (2x, 2y) = (6,8) in here.

[6.000009999951316, 8.00000999952033]


### **Using the gradient**
We now use the gradient to find the minimum of a function. 
For a function, 
$$ f(v) = \sum_{i} v_i^2
$$
the minmum occurs at $ v = [0,0,\cdots, 0]$ because every squared term is non-negative.<br>
The gradient is
$$ \Delta f(v) = [2v_1, 2v_2, \cdots, 2v_n]
$$
- The gradient points in the direction of increasing function.
- To minimise the gradient, we move in the opposite direction.
- Repeat the steps, until the changes get very small.


In [9]:
def sum_of_squares_gradient(v):
    return [ 2* v_i for v_i in v]

def step(v, direction, step_size):
    return [
        v_i + step_size*direction_i
        for v_i, direction_i in zip(v, direction)
    ]

v = [3, 4]

print(sum_of_squares_gradient(v))
# [6, 8]

print(step(v, sum_of_squares_gradient(v), -0.1)) # negative step size so we move against the gradient.
# [2.4, 3.2]

[6, 8]
[2.4, 3.2]


In [13]:
#Iteratively finding the minimum.Instead of taking one step, we keep updating v until the change is smaller than a chosen tolerance.
import random

def distance(v, w):
    return sum((v_i - w_i) ** 2
               for v_i, w_i in zip(v, w)) ** 0.5

v = [random.randint(-10, 10) for _ in range(3)] #random point in three dimensions.
tolerance = 0.0000001

while True:
    gradient = sum_of_squares_gradient(v)
    next_v = step(v, gradient, -0.01) #move in the opp dir of gradient

    if distance(next_v, v) < tolerance: #chcek whether change is sufficiently small.
        break

    v = next_v #reoeat the process

print(v)


[9.010858131342753e-07, 4.505429065671381e-06, 1.8021716262685506e-06]


### **Choosing the right step size**
The step size determines how far we move in the opposite direction of the gradient at each iteration. We took 0.01 as our step size. Choosing it carefully matters because:
- **Too small**: convergence is slow, and we need many iterations.
- **Too large**: we may overshoot the minimum and jump back and forth, or even move farther away from it.
- **A suitable step size** : helps us approach the minimum efficiently.

In [15]:
def gradient_descent(step_size, iterations=20):
    v = [10.0]

    for _ in range(iterations):
        gradient = sum_of_squares_gradient(v)
        v = step(v, gradient, -step_size)

    return v[0]

for size in [0.001, 0.01, 0.1, 0.6, 1.1]:
    print(size, "-", gradient_descent(size))

0.001 - 9.607509570263426
0.01 - 6.676079717550945
0.1 - 0.11529215046068471
0.6 - 1.0485759999999954e-13
1.1 - 383.3759992447485


## **Batch Gradient Descent**
Batch gradient descent uses the gradient of the function to update the current point repeatedly until it converges.
- At every iteration:
1. Calculate the gradient at the current point.
2. Move in the opposite direction.
3. Check whether the new point is sufficiently close to the previous point.
4. Stop when the change is smaller than the tolerance.

- Make the step size selection automatic
Instead of choosing one fixed step size, the book tries several candidates and chooses the one that gives the smallest value of the objective function.


##